# 🤖 TRUST Agents — Vietnamese Fake News Detection on Colab (GPU)

<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 20px; border-radius: 12px; color: white; margin-bottom: 20px;">
<h2>🚀 Full Pipeline: Claim Extraction → Evidence Retrieval → Verification → Explanation</h2>
<p>Running on <b>FREE T4 GPU</b> • Gemini/Groq LLM • FAISS GPU Index • ViFactCheck Dataset</p>
</div>

**Features:**
- 🔍 Claim Extraction (NER + Dependency Parsing + LLM)
- 📚 Hybrid RAG (FAISS GPU + fallback web search)
- ⚖️ LLM Verification với citation validation
- 📝 Vietnamese NLP (underthesea)
- 💾 Persistent FAISS index on Google Drive

**Runtime:** Runtime → Change runtime type → **T4 GPU**

---


In [ ]:
# ============================================================
# CELL 1: Setup & Installation
# ============================================================
from google.colab import drive
drive.mount("/content/drive")
print("✅ Google Drive mounted")

!pip install -q --upgrade pip
!pip install -q faiss-gpu
!pip install -q torch transformers sentence-transformers[torch] datasets accelerate numpy pandas
!pip install -q google-genai groq langchain langchain-google-genai langgraph
!pip install -q underthesea

# Try editable install from Drive
!pip install -q -e /content/drive/My\ Drive/trust_agents 2>/dev/null || \
  !pip install -q -e . 2>/dev/null || \
  echo "⚠️  Project not found locally, using inline imports"

print("✅ All dependencies installed")


In [ ]:
# ============================================================
# CELL 2: GPU & Environment Check
# ============================================================
import torch
import faiss
import importlib

print("=" * 60)
print("🖥️  GPU & Hardware Info")
print("=" * 60)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    compute_cap = torch.cuda.get_device_capability(0)
    print("✅ GPU: " + gpu_name)
    print("   Memory: {:.1f} GB".format(gpu_memory))
    print("   Compute: {}.{}".format(compute_cap[0], compute_cap[1]))
    print("   CUDA: " + str(torch.version.cuda))
    _ = torch.randn(1000, 1000, device="cuda")
    torch.cuda.synchronize()
    print("   GPU compute test: ✅ PASSED")
else:
    print("❌ No GPU detected!")

print()
print("=" * 60)
print("🔍 FAISS Info")
print("=" * 60)
print("FAISS version: " + faiss.__version__)
has_gpu = hasattr(faiss, "StandardGpuResources")
print("GPU support: " + ("✅ YES" if has_gpu else "❌ NO"))

if has_gpu:
    try:
        res = faiss.StandardGpuResources()
        print("FAISS GPU resources: ✅ Available")
        del res
    except Exception as e:
        print("FAISS GPU resources: ⚠️  " + str(e))
else:
    print("FAISS GPU resources: ❌ Not available (will use CPU)")

print()
print("=" * 60)
print("📦 Package Versions")
print("=" * 60)
print("PyTorch: " + torch.__version__)
print("CUDA available: " + str(torch.cuda.is_available()))
print("Transformers: " + importlib.import_module("transformers").__version__)
print("SentenceTransformers: " + importlib.import_module("sentence_transformers").__version__)


In [ ]:
# ============================================================
# CELL 3: Google Drive & FAISS Index Setup
# ============================================================
import os, time, pickle, numpy as np
from pathlib import Path
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

DRIVE_BASE = Path("/content/drive/MyDrive")
TRUST_DIR = DRIVE_BASE / "trust_agents"
INDEX_DIR = TRUST_DIR / "data" / "faiss_index"
TRUST_DIR.mkdir(parents=True, exist_ok=True)
INDEX_DIR.mkdir(parents=True, exist_ok=True)
print("📁 Index directory: " + str(INDEX_DIR))

index_file = INDEX_DIR / "index.faiss"
docs_file = INDEX_DIR / "documents.pkl"
documents = None
embed_model = None

if index_file.exists() and docs_file.exists():
    print("✅ FAISS index found on Drive — loading...")
    index = faiss.read_index(str(index_file))
    with open(docs_file, "rb") as f:
        documents = pickle.load(f)
    print("   Loaded {} vectors, {} documents".format(index.ntotal, len(documents)))
    print("   Loading embedding model...")
    embed_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    if torch.cuda.is_available():
        embed_model = embed_model.to("cuda")
    print("   FAISS GPU index: ✅")
else:
    print("🆕 Building FAISS index from ViFactCheck dataset...")
    print("   (This runs once per Drive; subsequent runs load from cache)")

    print("\n📥 Downloading ViFactCheck dataset...")
    start = time.time()
    dataset = load_dataset("tranthaihoa/vifactcheck", split="train")
    print("   Downloaded {} samples in {:.1f}s".format(len(dataset), time.time()-start))

    print("\n🤖 Loading embedding model...")
    start = time.time()
    embed_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    if torch.cuda.is_available():
        embed_model = embed_model.to("cuda")
        try:
            gpu_name = torch.cuda.get_device_name(0)
            print("   Model on GPU: " + gpu_name)
        except:
            print("   Model on GPU: ✅")
    print("   Loaded in {:.1f}s".format(time.time()-start))

    MAX_SAMPLES = 10000
    documents = []
    texts = []
    for i, item in enumerate(dataset):
        if i >= MAX_SAMPLES:
            break
        content = str(item.get("claim","")) + " " + str(item.get("evidence",""))
        content = content.strip().replace("\n"," ")
        if content:
            texts.append(content)
            documents.append({
                "content": content,
                "claim": item.get("claim",""),
                "evidence": item.get("evidence",""),
                "label": item.get("label",""),
                "source": item.get("source","ViFactCheck"),
            })

    print("   Prepared {} documents".format(len(documents)))

    print("\n⚡ Building FAISS GPU index...")
    start = time.time()
    embeddings = embed_model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
    faiss.normalize_L2(embeddings)
    print("   Embeddings: " + str(embeddings.shape))
    dim = embeddings.shape[1]

    if hasattr(faiss, "StandardGpuResources"):
        cpu_index = faiss.IndexFlatIP(dim)
        cpu_index.add(embeddings.astype(np.float32))
        gpu_res = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(gpu_res, 0, cpu_index)
        print("   Index on GPU: ✅")
    else:
        index = faiss.IndexFlatIP(dim)
        index.add(embeddings.astype(np.float32))
        print("   Index on CPU: fallback")

    print("   Index built in {:.1f}s".format(time.time()-start))
    print("   Total vectors: " + str(index.ntotal))

    print("\n💾 Saving index to Google Drive...")
    if hasattr(index, "to_cpu"):
        cpu_index_save = faiss.index_gpu_to_cpu(index)
    else:
        cpu_index_save = index
    faiss.write_index(cpu_index_save, str(index_file))
    with open(docs_file, "wb") as f:
        pickle.dump(documents, f)
    print("   Saved to " + str(INDEX_DIR))
    print("   File size: {:.1f} MB".format(index_file.stat().st_size / 1e6))

print("✅ FAISS index ready")
embed_model_global = embed_model
documents_global = documents
index_global = index


In [ ]:
# ============================================================
# CELL 4: LLM Configuration (Multi-source Secret Handling)
# ============================================================
import os, json, getpass
from IPython.display import clear_output, display, HTML

try:
    from google.colab import userdata
    HAS_USERDATA = True
except ImportError:
    HAS_USERDATA = False

llm_mode = None
llm_client = None

def get_secret(key):
    """Get secret from Colab Secrets, then Env, then Prompt."""
    # 1. Try Colab Secrets (userdata)
    if HAS_USERDATA:
        try:
            val = userdata.get(key)
            if val: return val
        except:
            pass
    
    # 2. Try os.environ
    val = os.environ.get(key)
    if val: return val
    
    # 3. Prompt user
    try:
        val = getpass.getpass(f"🔑 Enter {key} (or press Enter to skip):
")
        return val
    except:
        return ""

print("=" * 60)
print("🤖 LLM Configuration")
print("=" * 60)

# --- Try Gemini ---
gemini_key = get_secret("GEMINI_API_KEY") or get_secret("GOOGLE_API_KEY")

if gemini_key:
    try:
        import google.genai as genai
        genai.configure(api_key=gemini_key)
        model = genai.GenerativeModel("gemini-2.0-flash")
        # Quick test
        _ = model.generate_content("Hello", generation_config=genai.types.GenerateContentConfig(max_output_tokens=10))
        print("✅ Google Gemini: Connected")
        llm_mode = "gemini"
        llm_client = {"key": gemini_key, "model": "gemini-2.0-flash"}
    except Exception as e:
        print(f"⚠️  Gemini failed: {e}")

# --- Try Groq ---
if not llm_mode:
    groq_key = get_secret("GROQ_API_KEY")
    if groq_key:
        try:
            from groq import Groq
            client = Groq(api_key=groq_key)
            resp = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": "hello"}],
                max_tokens=10,
            )
            print("✅ Groq (Llama-3.3-70B): Connected")
            llm_mode = "groq"
            llm_client = {"key": groq_key}
        except Exception as e:
            print(f"⚠️  Groq failed: {e}")

# --- Mock Mode ---
if not llm_mode:
    print("⚠️  No API key found in Secrets, Env, or Prompt.")
    print("   Using MOCK LLM (demo mode) — results will be heuristic-based.")
    llm_mode = "mock"

print()
print(f"📌 LLM Mode: {llm_mode.upper()}")

# ── Wrapper Functions ──
def call_llm(prompt, system_prompt="", max_tokens=1024):
    if llm_mode == "gemini":
        import google.genai as genai
        genai.configure(api_key=llm_client["key"])
        model = genai.GenerativeModel(llm_client["model"])
        full = system_prompt + "

" + prompt if system_prompt else prompt
        resp = model.generate_content(full, generation_config=genai.types.GenerateContentConfig(temperature=0.1, max_output_tokens=max_tokens))
        return resp.text.strip() if hasattr(resp, "text") else str(resp)
    elif llm_mode == "groq":
        from groq import Groq
        client = Groq(api_key=llm_client["key"])
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": prompt})
        resp = client.chat.completions.create(model="llama-3.3-70b-versatile", messages=messages, temperature=0.1, max_tokens=max_tokens)
        return resp.choices[0].message.content.strip()
    else:
        return mock_llm_response(prompt)

def call_llm_json(prompt, system_prompt="", max_tokens=1024):
    import re
    response = call_llm(prompt, system_prompt, max_tokens)
    match = re.search(r"\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}", response, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except:
            pass
    raise ValueError("Could not parse JSON from: " + response[:200])

# ── Mock LLM (rule-based demo) ──
FAKE_KEYWORDS = ["alien", "người ngoài hành tinh", "ufo", "50%", "chữa khỏi ung thư", "che giấu", "tin đồn", "giải cứu"]
REAL_KEYWORDS = ["world bank", "bộ y tế", "5.6%", "2023", "báo cáo", "official", "confirmed"]

def mock_llm_response(prompt):
    p = prompt.lower()
    if any(kw in p for kw in FAKE_KEYWORDS):
        return json.dumps({"verdict": "false", "confidence": 0.85, "reasoning": "Claim contains fake news indicators"})
    if any(kw in p for kw in REAL_KEYWORDS):
        return json.dumps({"verdict": "true", "confidence": 0.85, "reasoning": "Claim matches known facts from official sources"})
    return json.dumps({"verdict": "uncertain", "confidence": 0.3, "reasoning": "Insufficient context for determination"})

print("✅ LLM wrapper ready")


In [ ]:
# ============================================================
# CELL 5: Initialize TRUST Pipeline
# ============================================================
import re, time, logging
from typing import List, Dict, Any

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
print("=" * 60)
print("🔧 TRUST Agents Pipeline — Initialization")
print("=" * 60)

CLAIM_EXTRACTION_PROMPT = """Bạn là một chuyên gia trích xuất thông tin thực từ văn bản tiếng Việt.
Từ văn bản đầu vào, hãy trích xuất TẤT CẢ các tuyên bố thực tế (factual claims).
QUY TẮC:
1. Chỉ trích xuất các tuyên bố thực tế, KHÔNG phải ý kiến
2. Mỗi tuyên bố phải độc lập và có thể xác minh riêng
3. Ưu tiên các con số, ngày tháng, sự kiện cụ thể
4. Trả về JSON: {"claims": ["claim1", "claim2", ...]}
Văn bản: {text}
Trả lời (chỉ JSON):"""

def extract_claims(text: str) -> List[str]:
    prompt = CLAIM_EXTRACTION_PROMPT.format(text=text)
    try:
        result = call_llm_json(prompt)
        claims = result.get("claims", [])
        if isinstance(claims, list):
            return [str(c).strip() for c in claims if c]
    except Exception as e:
        print("⚠️  Claim extraction failed: " + str(e))
    return []

def retrieve_evidence(query: str, k: int = 5) -> List[Dict[str, Any]]:
    if index_global.ntotal == 0:
        return []
    q_emb = embed_model_global.encode([query])
    import faiss
    faiss.normalize_L2(q_emb)
    k = min(k, index_global.ntotal)
    scores, indices = index_global.search(q_emb.astype(np.float32), k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < len(documents_global):
            doc = documents_global[idx].copy()
            doc["score"] = float(score)
            results.append(doc)
    return results

VERIFIER_PROMPT = """Bạn là chuyên gia kiểm chứng thông tin (fact-checker).
Dựa trên bằng chứng (evidence) được cung cấp, hãy xác định tuyên bố (claim) là:
- TRUE: Bằng chứng hỗ trợ tuyên bố
- FALSE: Bằng chứng bác bỏ tuyên bố
- UNCERTAIN: Không đủ bằng chứng để kết luận
Claim: {claim}
Evidence:
{evidence}
Trả về JSON: {{"verdict": "true|false|uncertain", "confidence": 0.0-1.0, "reasoning": "..."}}
Chỉ trả về JSON."""

def verify_claim(claim: str, evidence: List[Dict]) -> Dict[str, Any]:
    if not evidence:
        return {"verdict": "uncertain", "confidence": 0.1, "reasoning": "No evidence available"}
    evidence_text = "\n".join("[{}] {}".format(doc.get("source","unknown"), doc.get("content","")[:300]) for doc in evidence[:3])
    prompt = VERIFIER_PROMPT.format(claim=claim, evidence=evidence_text)
    try:
        result = call_llm_json(prompt)
        verdict = result.get("verdict", "uncertain").lower()
        conf = result.get("confidence", 0.5)
        if isinstance(conf, (int, float)) and conf > 1:
            conf = conf / 100.0
        conf = max(0.0, min(1.0, float(conf)))
        return {
            "verdict": verdict if verdict in ("true", "false", "uncertain") else "uncertain",
            "confidence": conf,
            "reasoning": result.get("reasoning", ""),
        }
    except Exception as e:
        print("⚠️  Verification failed: " + str(e))
        return {"verdict": "uncertain", "confidence": 0.1, "reasoning": "Error: " + str(e)}

EXPLAINER_PROMPT = """Bạn là chuyên gia giải thích kết quả kiểm chứng.
Claim: {claim}
Verdict: {verdict} ({confidence})
Reasoning: {reasoning}
Trả về JSON: {{"summary": "...", "explanation": "..."}}"""

def explain_claim(claim: str, verdict_data: Dict, evidence: List[Dict]) -> Dict[str, Any]:
    prompt = EXPLAINER_PROMPT.format(
        claim=claim,
        verdict=verdict_data.get("verdict", "uncertain"),
        confidence="{:.0f}%".format(verdict_data.get("confidence", 0)*100),
        reasoning=verdict_data.get("reasoning", ""),
    )
    try:
        result = call_llm_json(prompt)
        return {
            "summary": result.get("summary", verdict_data.get("verdict", "").upper()),
            "explanation": result.get("explanation", verdict_data.get("reasoning", "")),
            "verdict": verdict_data.get("verdict", "uncertain"),
            "confidence": verdict_data.get("confidence", 0),
            "evidence_count": len(evidence),
        }
    except Exception:
        return {
            "summary": verdict_data.get("verdict", "uncertain").upper(),
            "explanation": verdict_data.get("reasoning", ""),
            "verdict": verdict_data.get("verdict", "uncertain"),
            "confidence": verdict_data.get("confidence", 0),
            "evidence_count": len(evidence),
        }

def fact_check(text: str, top_k: int = 5) -> Dict[str, Any]:
    print("\n📝 Input: " + text[:100] + ("..." if len(text) > 100 else ""))
    start = time.time()

    print("\n[1/4] 🔍 Extracting claims...")
    claims = extract_claims(text)
    if not claims:
        return {
            "original_text": text,
            "claims": [],
            "results": [],
            "summary": {"status": "no_claims", "message": "No claims found"},
        }
    print("    Found {} claim(s)".format(len(claims)))

    results = []
    for i, claim in enumerate(claims, 1):
        print("\n[2/4] 📚 Retrieving evidence for claim {}/{}...".format(i, len(claims)))
        evidence = retrieve_evidence(claim, k=top_k)
        print("    Found {} evidence passage(s)".format(len(evidence)))

        print("[3/4] ⚖️  Verifying claim...")
        verdict = verify_claim(claim, evidence)
        print("    Verdict: {} ({:.0f}%)".format(verdict.get("verdict","?").upper(), verdict.get("confidence",0)*100))

        print("[4/4] 📝 Generating explanation...")
        report = explain_claim(claim, verdict, evidence)
        report["claim"] = claim
        report["top_evidence"] = [{"content": e.get("content","")[:150], "score": e.get("score",0)} for e in evidence[:3]]
        results.append(report)

    elapsed = time.time() - start
    verdicts = {"true": 0, "false": 0, "uncertain": 0}
    confidences = []
    for r in results:
        v = r.get("verdict", "uncertain")
        if v in verdicts:
            verdicts[v] += 1
        confidences.append(r.get("confidence", 0))

    summary = {
        "total_claims": len(results),
        "verdicts": verdicts,
        "average_confidence": round(sum(confidences)/len(confidences), 3) if confidences else 0,
        "elapsed_seconds": round(elapsed, 2),
    }
    print("\n✅ Pipeline complete in {:.2f}s".format(elapsed))
    return {"original_text": text, "claims": claims, "results": results, "summary": summary}

print("✅ Pipeline components ready")


In [ ]:
# ============================================================
# CELL 6: Demo — Fact Check Samples
# ============================================================
from IPython.display import display, HTML

VCOLORS = {"true": ("#22c55e","#bbf7d0"), "false": ("#ef4444","#fecaca"), "uncertain": ("#f59e0b","#fef3c7")}
VLABELS = {"true": "✅ TRUE (Thật)", "false": "❌ FALSE (Giả)", "uncertain": "⚠️  UNCERTAIN (Không thể xác minh)"}

def display_result(result):
    summary = result.get("summary", {})
    verdicts = summary.get("verdicts", {})
    html = '''<div style="font-family: system-ui, sans-serif; max-width: 900px; margin: auto;">
<div style="display: flex; gap: 16px; margin-bottom: 24px; padding: 16px; background: #f8fafc; border-radius: 10px; border: 1px solid #e2e8f0;">
  <div style="flex: 1; text-align: center;"><div style="font-size: 28px; font-weight: 700; color: #1e293b;">''' + str(summary.get("total_claims",0)) + '''</div><div style="font-size: 13px; color: #64748b;">Claims</div></div>
  <div style="flex: 1; text-align: center; border-left: 1px solid #e2e8f0;"><div style="font-size: 28px; font-weight: 700; color: #22c55e;">''' + str(verdicts.get("true",0)) + '''</div><div style="font-size: 13px; color: #64748b;">✅ True</div></div>
  <div style="flex: 1; text-align: center; border-left: 1px solid #e2e8f0;"><div style="font-size: 28px; font-weight: 700; color: #ef4444;">''' + str(verdicts.get("false",0)) + '''</div><div style="font-size: 13px; color: #64748b;">❌ False</div></div>
  <div style="flex: 1; text-align: center; border-left: 1px solid #e2e8f0;"><div style="font-size: 28px; font-weight: 700; color: #f59e0b;">''' + str(verdicts.get("uncertain",0)) + '''</div><div style="font-size: 13px; color: #64748b;">⚠️ Uncertain</div></div>
  <div style="flex: 1; text-align: center; border-left: 1px solid #e2e8f0;"><div style="font-size: 28px; font-weight: 700; color: #64748b;">''' + "{:.0f}%".format(summary.get("average_confidence",0)*100) + '''</div><div style="font-size: 13px; color: #64748b;">Avg Confidence</div></div>
  <div style="flex: 1; text-align: center; border-left: 1px solid #e2e8f0;"><div style="font-size: 28px; font-weight: 700; color: #64748b;">''' + "{:.1f}s".format(summary.get("elapsed_seconds",0)) + '''</div><div style="font-size: 13px; color: #64748b;">Time</div></div>
</div>'''
    for r in result.get("results", []):
        v = r.get("verdict", "uncertain")
        bc, bdr = VCOLORS.get(v, ("#94a3b8","#f1f5f9"))
        lbl = VLABELS.get(v, v.upper())
        conf = r.get("confidence", 0)
        claim = r.get("claim","")
        explanation = r.get("explanation","")
        ev_html = ""
        for j, ev in enumerate(r.get("top_evidence",[]), 1):
            ev_html += '<div style="margin-top: 8px; padding: 8px 12px; background: #fff; border-radius: 6px; border-left: 3px solid #94a3b8;"><span style="font-size: 11px; font-weight: 600; color: #64748b;">#{j} ({score:.0f}%)</span> <span style="font-size: 13px; color: #334155;">{content}...</span></div>'.format(j=j, score=ev.get("score",0)*100, content=ev.get("content","")[:200])
        html += '<div style="margin-bottom: 20px; padding: 20px; background: {bc}; border-radius: 12px; border-left: 5px solid {bdr}; box-shadow: 0 1px 3px rgba(0,0,0,0.1);">'.format(bc=bc, bdr=bdr)
        html += '<div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 12px;"><div style="font-size: 22px; font-weight: 700; color: {bdr};">{lbl}</div><div style="font-size: 16px; font-weight: 600; color: #64748b;">{conf:.0f}% conf</div></div>'.format(bdr=bdr, lbl=lbl, conf=conf*100)
        html += '<div style="margin-bottom: 12px;"><div style="font-size: 13px; font-weight: 700; color: #475569; text-transform: uppercase; letter-spacing: 0.5px; margin-bottom: 4px;">📌 CLAIM</div><div style="font-size: 15px; color: #1e293b; line-height: 1.6;">{claim}</div></div>'.format(claim=claim)
        html += '<div style="margin-bottom: 12px;"><div style="font-size: 13px; font-weight: 700; color: #475569; text-transform: uppercase; letter-spacing: 0.5px; margin-bottom: 4px;">💡 EXPLANATION</div><div style="font-size: 14px; color: #334155; line-height: 1.6;">{exp}</div></div>'.format(exp=explanation)
        html += '<div><div style="font-size: 13px; font-weight: 700; color: #475569; text-transform: uppercase; letter-spacing: 0.5px; margin-bottom: 8px;">📚 TOP EVIDENCE</div>{ev}</div></div>'.format(ev=ev_html)
    html += "</div>"
    display(HTML(html))

print("=" * 60)
print("🎯 Demo: Fact Check Vietnamese News")
print("=" * 60)

FAKE_NEWS = "Việt Nam đã phát hiện người ngoài hành tinh tại Hà Nội và chính phủ đang che giấu thông tin này."
REAL_NEWS = "Theo báo cáo của World Bank, Việt Nam đạt tăng trưởng GDP 5.6% trong năm 2023, thuộc nhóm cao nhất ASEAN."
UNVERIFIABLE = "Một nguồn tin cho biết lãnh đạo đất nước sẽ có cuộc họp quan trọng vào tuần tới nhưng không tiết lộ chi tiết."

samples = [
    ("TIN GIẢ — UFO", FAKE_NEWS),
    ("TIN THẬT — GDP", REAL_NEWS),
    ("KHÔNG THỂ XÁC MINH — NGUỒN TIN ẨN DANH", UNVERIFIABLE),
]

for title, text in samples:
    print("\n\n" + "=" * 60)
    print("  " + title)
    print("=" * 60)
    print("Text: " + text)
    print()
    result = fact_check(text)
    display_result(result)


In [ ]:
# ============================================================
# CELL 7: Interactive Demo
# ============================================================
user_text = ''' '''
# Paste your Vietnamese news text above in user_text variable
# Example: user_text = "Việt Nam đạt tăng trưởng GDP 8% trong quý 1 năm 2024."

if user_text.strip() and "Example" not in user_text:
    print("📝 Input text:")
    print(user_text.strip())
    print()
    result = fact_check(user_text.strip())
    display_result(result)
else:
    display(HTML('<div style="padding: 20px; background: #fffbeb; border-radius: 10px; border: 1px solid #fbbf24;"><h3>🎯 Interactive Mode</h3><p>Edit the <code>user_text</code> variable above with your Vietnamese news text, then run this cell.</p><p>Try: <code>Việt Nam phát hiện người ngoài hành tinh tại Hà Nội</code></p></div>'))
    demo_text = "Việt Nam phát hiện người ngoài hành tinh tại Hà Nội và chính phủ đang che giấu thông tin."
    print("Auto-demo: " + demo_text)
    print()
    result = fact_check(demo_text)
    display_result(result)


In [ ]:
# ============================================================
# CELL 8: Batch Testing & Statistics
# ============================================================
from collections import Counter

print("=" * 60)
print("📊 Batch Testing — ViFactCheck Test Set")
print("=" * 60)

print("\n📥 Loading test samples...")
test_dataset = load_dataset("tranthaihoa/vifactcheck", split="test")
print("   Test samples: " + str(len(test_dataset)))

BATCH_SIZE = 20
batch = []
seen_labels = {"true": 0, "false": 0, "unverified": 0}
for item in test_dataset:
    label = item.get("label","")
    if label in seen_labels and seen_labels[label] < BATCH_SIZE // 3:
        batch.append(item)
        seen_labels[label] += 1
    if len(batch) >= BATCH_SIZE:
        break

print("   Selected " + str(len(batch)) + " diverse samples")

print("\n⚡ Running fact-check on batch...\n")

predictions = []
ground_truth = []
times = []

for i, item in enumerate(batch):
    text = item.get("claim","")
    true_label = item.get("label","unverified")
    print("[{}/{}] {}".format(i+1, len(batch), text[:60] + ("..." if len(text)>60 else "")))
    start = time.time()
    result = fact_check(text)
    elapsed = time.time() - start
    times.append(elapsed)
    pred = result.get("results",[{}])[0].get("verdict","uncertain") if result.get("results") else "uncertain"
    predictions.append(pred)
    ground_truth.append(true_label)
    print("   -> Predicted: {}, Expected: {}, Time: {:.1f}s".format(pred.upper(), true_label, elapsed))

print("\n" + "=" * 60)
print("📈 Results Summary")
print("=" * 60)
correct = sum(1 for p,g in zip(predictions,ground_truth) if p==g)
accuracy = correct / len(predictions) * 100
print("Accuracy: {:.1f}% ({}/{})".format(accuracy, correct, len(predictions)))
print("Avg time per sample: {:.1f}s".format(sum(times)/len(times)))
print("Min: {:.1f}s, Max: {:.1f}s".format(min(times), max(times)))
print("\nPrediction distribution:")
for v,c in Counter(predictions).most_common():
    print("  {}: {}".format(v.upper(), c))
print("\nGround truth distribution:")
for v,c in Counter(ground_truth).most_common():
    print("  {}: {}".format(v, c))


In [ ]:
# ============================================================
# CELL 9: Performance Benchmark — GPU vs CPU
# ============================================================
import time

print("=" * 60)
print("⚡ GPU Performance Benchmark")
print("=" * 60)

BENCHMARK_TEXT = "Việt Nam đạt tăng trưởng GDP 5.6% trong năm 2023 theo báo cáo của World Bank, thuộc nhóm cao nhất ASEAN. Bộ Y tế công bố số ca mắc COVID-19 giảm 30%. Các chuyên gia dự đoán kinh tế sẽ tiếp tục tăng trưởng mạnh."
N_RUNS = 3

print("\n[1/3] 📊 Embedding Speed (GPU vs CPU)")
print("-" * 40)
for device in ["cuda", "cpu"]:
    if device == "cpu" and not torch.cuda.is_available():
        print("CPU: (skipped - no GPU fallback needed)")
        continue
    model_bench = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    if device == "cuda":
        model_bench = model_bench.to("cuda")
    times = []
    for _ in range(N_RUNS):
        start = time.time()
        _ = model_bench.encode([BENCHMARK_TEXT] * 10)
        if device == "cuda":
            torch.cuda.synchronize()
        times.append(time.time() - start)
    avg = sum(times) / len(times)
    label = "GPU (CUDA)" if device == "cuda" else "CPU"
    print("{}: {:.1f}ms avg ({} runs)".format(label, avg*1000, len(times)))

print("\n[2/3] 🔍 FAISS Search Speed")
print("-" * 40)
if index_global.ntotal > 0:
    for device_label, idx in [("GPU", index_global), ("CPU", None)]:
        if idx is None:
            continue
        times = []
        for _ in range(N_RUNS):
            start = time.time()
            for _ in range(10):
                q_emb = embed_model_global.encode([BENCHMARK_TEXT])
                import faiss
                faiss.normalize_L2(q_emb)
                _ = idx.search(q_emb.astype(np.float32), 5)
            if device_label == "GPU":
                torch.cuda.synchronize()
            times.append(time.time() - start)
        avg = sum(times) / len(times)
        print("FAISS {}: {:.1f}ms avg for 10 queries".format(device_label, avg*1000))

print("\n[3/3] 🔄 End-to-End Pipeline Timing")
print("-" * 40)
for i in range(N_RUNS):
    start = time.time()
    result = fact_check(BENCHMARK_TEXT)
    elapsed = time.time() - start
    n_claims = len(result["results"])
    verd = result["results"][0].get("verdict","?").upper() if result["results"] else "?"
    print("Run {}: {:.2f}s — Claims: {}, Verdict: {}".format(i+1, elapsed, n_claims, verd))

print("\n" + "=" * 60)
print("💾 GPU Memory")
print("=" * 60)
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / 1e9
    reserved = torch.cuda.memory_reserved(0) / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print("Allocated: {:.2f} GB".format(allocated))
    print("Reserved:  {:.2f} GB".format(reserved))
    print("Total:     {:.2f} GB".format(total))
    print("Utilization: {:.1f}%".format(allocated/total*100))

print("\n✅ Benchmark complete!")
